In [1]:
!pip install --quiet tabicl[forecast]

import numpy as np
import pandas as pd
import gc, warnings, os, time
from tabicl import TabICLRegressor

warnings.filterwarnings('ignore')

T0 = time.time()

TRAIN_PATH = '/kaggle/input/competitions/ts-forecasting/train.parquet'
TEST_PATH = '/kaggle/input/competitions/ts-forecasting/test.parquet'
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH, TEST_PATH = 'train.parquet', 'test.parquet'

VAL_THRESHOLD = 3500
FORECAST_WINDOWS = [1]  # 只训练horizon 1

# ── 极限配置 ──
N_ESTIMATORS = 4  # 最小集成
BATCH_SIZE = 32   # 适中的batch,避免内存溢出
RANDOM_STATE = 777
seeds = [777]     # 单种子
SAMPLE_RATIO = 0.001  # 只用0.1%数据训练

print(f'🏆 TabICL v2.0.3: MINIMAL MODE (5% data, 0 features)')

# 定义所有特征列
FEATURE_COLS = ['feature_a','feature_b','feature_c','feature_d','feature_e','feature_f','feature_g','feature_h','feature_i','feature_j','feature_k',
 'feature_l','feature_m','feature_n','feature_o','feature_p','feature_q','feature_r','feature_s','feature_t','feature_u','feature_v','feature_w',
 'feature_x','feature_y','feature_z','feature_aa','feature_ab','feature_ac','feature_ad','feature_ae','feature_af','feature_ag','feature_ah','feature_ai',
 'feature_aj','feature_ak','feature_al','feature_am','feature_an','feature_ao','feature_ap','feature_aq','feature_ar','feature_as','feature_at',
 'feature_au','feature_av','feature_aw','feature_ax','feature_ay','feature_az','feature_ba','feature_bb','feature_bc','feature_bd','feature_be',
 'feature_bf','feature_bg','feature_bh','feature_bi','feature_bj','feature_bk','feature_bl','feature_bm','feature_bn','feature_bo','feature_bp',
 'feature_bq','feature_br','feature_bs','feature_bt','feature_bu','feature_bv','feature_bw','feature_bx','feature_by','feature_bz','feature_ca',
 'feature_cb','feature_cc','feature_cd','feature_ce','feature_cf','feature_cg','feature_ch']

# Compute encoding stats (只计算必要的)
print('Computing stats...')
temp = pd.read_parquet(TRAIN_PATH, columns=['sub_category', 'sub_code', 'y_target', 'ts_index'])
train_only = temp[temp.ts_index <= VAL_THRESHOLD]
train_stats = {
    'sub_category': train_only.groupby('sub_category')['y_target'].mean().to_dict(),
    'sub_code': train_only.groupby('sub_code')['y_target'].mean().to_dict(),
    'global_mean': train_only['y_target'].mean()
}
del temp, train_only; gc.collect()

def build_features_minimal(data):
    """极限特征工程 - 只保留绝对必要的基础特征"""
    df = data.copy()
    
    # 只保留原始特征 + 最简单的编码
    for c in ['sub_category', 'sub_code']:
        df[c + '_enc'] = df[c].map(train_stats[c]).fillna(train_stats['global_mean'])
    
    ts_idx = df['ts_index'].values
    df['t_sin'] = np.sin(2 * np.pi * ts_idx / 100)
    df['t_cos'] = np.cos(2 * np.pi * ts_idx / 100)
    del ts_idx
    
    # 保留训练集需要的所有列(包括y_target和weight)
    essential_cols = ['id', 'ts_index', 'horizon', 'code', 'sub_code', 'sub_category']
    target_cols = ['y_target', 'weight']  # 训练集需要
    feature_cols = ['feature_al', 'feature_am', 'feature_cg', 'feature_by', 'feature_s',
                    'sub_category_enc', 'sub_code_enc', 't_sin', 't_cos']
    
    # 过滤存在的列
    keep_cols = essential_cols.copy()
    for col in target_cols:
        if col in df.columns:
            keep_cols.append(col)
    for col in feature_cols:
        if col in df.columns:
            keep_cols.append(col)
    
    df = df[keep_cols]
    
    return df.fillna(0)

def get_feature_columns(df):
    exclude = {'id', 'code', 'sub_code', 'sub_category', 'horizon',
               'ts_index', 'weight', 'y_target'}
    return [c for c in df.columns if c not in exclude]

def weighted_rmse_score(y_target, y_pred, w):
    y_target = np.array(y_target)
    y_pred = np.array(y_pred)
    w = np.array(w)
    denom = np.sum(w * (y_target ** 2))
    if denom <= 0: return 0.0
    numerator = np.sum(w * ((y_target - y_pred) ** 2))
    ratio = numerator / denom
    return float(np.sqrt(1.0 - np.clip(ratio, 0.0, 1.0)))

print('Ready')

def solve_horizon(horizon):
    t0 = time.time()
    print(f'\n{"="*60}')
    print(f'  🏆 HORIZON {horizon} (MINIMAL MODE)')
    print(f'{"="*60}')

    # Load with minimal columns
    tr = pd.read_parquet(TRAIN_PATH).query(f'horizon == {horizon}')
    te = pd.read_parquet(TEST_PATH)  # 测试集需要预测全部数据

    # 只保留id、特征列和预测列
    keep_cols = ['id'] + FEATURE_COLS + ['y_target', 'horizon', 'ts_index', 'weight']
    tr = tr[[c for c in keep_cols if c in tr.columns]]
    te_cols = ['id', 'horizon', 'ts_index'] + FEATURE_COLS
    te = te[[c for c in te_cols if c in te.columns]]

    print(f'  Data: {len(tr):,} train + {len(te):,} test')

    # Build minimal features
    # 直接使用原始特征,不进行额外特征工程
    tr = tr.fillna(0)
    gc.collect()
    te = te.fillna(0)

    # 直接使用所有特征列
    feats = FEATURE_COLS
    for c in feats:
        if c not in te.columns:
            te[c] = 0
    print(f'  Features: {len(feats)}')

    # Split
    fit_m = tr.ts_index <= VAL_THRESHOLD
    val_m = tr.ts_index > VAL_THRESHOLD

    X_fit = tr.loc[fit_m, feats].astype(np.float32)
    y_fit = tr.loc[fit_m, 'y_target'].values

    X_val = tr.loc[val_m, feats].astype(np.float32)
    y_val = tr.loc[val_m, 'y_target'].values
    w_val = tr.loc[val_m, 'weight'].values

    # 采样比例
    SAMPLE_RATIO = 0.001
    n_samples = int(len(X_fit) * SAMPLE_RATIO)
    indices = np.random.RandomState(42).choice(len(X_fit), n_samples, replace=False)
    X_fit_sample = X_fit.iloc[indices] if hasattr(X_fit, 'iloc') else X_fit[indices]
    y_fit_sample = y_fit[indices]

    # 删除原始数据,释放内存
    del tr, X_fit, y_fit
    gc.collect()

    print(f'  Sample: {n_samples:,}')
    print(f'  y_fit_sample range: [{y_fit_sample.min():.4f}, {y_fit_sample.max():.4f}]')

    # ── 单模型训练 ──
    for i, s in enumerate(seeds):
        print(f'  Seed {i+1}/{len(seeds)}...')

        # 训练模型
        model = TabICLRegressor(
            n_estimators=N_ESTIMATORS,
            batch_size=BATCH_SIZE,
            kv_cache=False,
            device='cpu',
            random_state=s,
            verbose=True,  # 开启verbose查看训练详情
            checkpoint_version="tabicl-regressor-v2-20260212.ckpt",
        )
        model.fit(X_fit_sample, y_fit_sample)

        # 预测验证集
        val_pred = model.predict(X_val)
        print(f'    val_pred range: [{val_pred.min():.4f}, {val_pred.max():.4f}]')

        # 分批预测测试集,避免内存溢出
        print(f'    Predicting test set in batches...')
        batch_size = 10000
        tst_pred = np.zeros(len(te), dtype=np.float32)
        for i in range(0, len(te), batch_size):
            end_idx = min(i + batch_size, len(te))
            batch_X = te.iloc[i:end_idx][feats].astype(np.float32)
            tst_pred[i:end_idx] = model.predict(batch_X)
            if (i // batch_size) % 10 == 0:
                print(f'      Processed {end_idx:,} / {len(te):,}')
            del batch_X
            gc.collect()
        print(f'    tst_pred range: [{tst_pred.min():.4f}, {tst_pred.max():.4f}]')
        del model
        gc.collect()

    # Winsorize (在删除y_val之前执行)
    q005, q995 = np.quantile(y_val, [0.005, 0.995])
    tst_pred = np.clip(tst_pred, q005, q995)

    score = weighted_rmse_score(y_val, val_pred, w_val)

    # 删除验证集和训练数据,释放内存
    del X_fit_sample, y_fit_sample, X_val, y_val
    elapsed = time.time() - t0
    print(f'  H{horizon} Score: {score:.5f} | {elapsed/60:.1f}min')

    ids = te['id'].values
    del te
    gc.collect()
    return tst_pred, ids, score

results = []
scores = {}

for h in FORECAST_WINDOWS:
    p, i, s = solve_horizon(h)
    results.append(pd.DataFrame({'id': i, 'prediction': p}))
    scores[h] = s

sub = pd.concat(results, ignore_index=True)
sub.to_csv('submission.csv', index=False)

total = time.time() - T0
print(f'\n{"="*60}')
print(f'  🏆 MINIMAL MODE — FINAL RESULTS')
print(f'{"="*60}')
print(f'  Features: ~8 (original only)')
print(f'  Data: 5% sampled')
print(f'  Ensembles: 1 seed × 1 estimator')
print(f'  {"─"*56}')
for h, s in scores.items():
    print(f'    H{h}: {s:.5f}')
print(f'  {"="*60}')
print(f'  SUBMISSION READY')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chex 0.1.91 requires toolz>=1.0.0, but you have toolz 0.12.1 which is incompatible.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


🏆 TabICL v2.0.3: MINIMAL MODE (5% data, 0 features)
Computing stats...


Ready

  🏆 HORIZON 1 (MINIMAL MODE)


  Data: 1,394,653 train + 1,447,107 test


...

      Processed 1,210,000 / 1,447,107


      Processed 1,310,000 / 1,447,107


      Processed 1,410,000 / 1,447,107


    tst_pred range: [-61.4316, 74.3865]
  H1 Score: 0.00000 | 82.0min



  🏆 MINIMAL MODE — FINAL RESULTS
  Features: ~8 (original only)
  Data: 5% sampled
  Ensembles: 1 seed × 1 estimator
  ────────────────────────────────────────────────────────
    H1: 0.00000
  SUBMISSION READY
